In [ ]:
# torch

In [ ]:
import pandas as pd

train = pd.read_csv("../../train.csv")
test = pd.read_csv("../../test.csv")
sub = pd.read_csv("../../sample_submission.csv")

In [2]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
import warnings

warnings.filterwarnings("ignore")

train = pd.read_csv("../../train.csv")
test = pd.read_csv("../../test.csv")
y = train["stress_score"]

num_cols = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

# ── M-1: NaN 행과 비NaN 행을 완전히 분리해서 분석 ──
nan_mask = train["mean_working"].isnull()
has_mask = ~nan_mask

print("[M-1] NaN 행 vs 비NaN 행 — Full Tree 각각 학습")

# 비NaN 행만
tr_has = train[has_mask].copy()
feats_has = num_cols + ["mean_working"]
X_has = tr_has[feats_has].values
y_has = tr_has["stress_score"].values

dt_has = DecisionTreeRegressor(max_depth=None, random_state=42)
dt_has.fit(X_has, y_has)
print(
    f"  비NaN 행 (n={len(tr_has)}): train MAE={mean_absolute_error(y_has, dt_has.predict(X_has)):.6f}"
)
print(f"  비NaN 행 R²={dt_has.score(X_has, y_has):.6f}")
print(f"  비NaN 행 리프 수: {dt_has.get_n_leaves()}")

# NaN 행만
tr_nan = train[nan_mask].copy()
X_nan = tr_nan[num_cols].values
y_nan = tr_nan["stress_score"].values

dt_nan = DecisionTreeRegressor(max_depth=None, random_state=42)
dt_nan.fit(X_nan, y_nan)
print(
    f"\n  NaN 행 (n={len(tr_nan)}): train MAE={mean_absolute_error(y_nan, dt_nan.predict(X_nan)):.6f}"
)
print(f"  NaN 행 R²={dt_nan.score(X_nan, y_nan):.6f}")
print(f"  NaN 행 리프 수: {dt_nan.get_n_leaves()}")

# ── M-2: 비NaN 행에서 mean_working을 제거하면 R² 얼마나 떨어지나? ──
print("\n[M-2] 비NaN 행에서 mean_working 제거 시 R²")
X_no_wk = tr_has[num_cols].values
dt_no = DecisionTreeRegressor(max_depth=None, random_state=42)
dt_no.fit(X_no_wk, y_has)
print(
    f"  mean_working 제거: R²={dt_no.score(X_no_wk, y_has):.6f}  "
    f"MAE={mean_absolute_error(y_has, dt_no.predict(X_no_wk)):.6f}"
)
print(f"  (R²=1.000이면 mean_working이 없어도 완전 결정됨)")
print(f"  (R²<1.000이면 mean_working이 공식의 핵심 인자)")

# ── M-3: 비NaN 행의 depth별 R² ──
print("\n[M-3] 비NaN 행 depth별 R² (mean_working 포함)")
for d in [5, 10, 15, 20, 25, 30, None]:
    dt2 = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt2.fit(X_has, y_has)
    r2 = dt2.score(X_has, y_has)
    mae = mean_absolute_error(y_has, np.clip(dt2.predict(X_has), 0, 1))
    label = str(d) if d else "None"
    print(f"  depth={label:5s}: R²={r2:.6f}  MAE={mae:.6f}")

# ── M-4: NaN 행의 depth별 R² ──
print("\n[M-4] NaN 행 depth별 R² (mean_working 없이 수치 8개만)")
for d in [5, 10, 15, 20, 25, 30, None]:
    dt3 = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt3.fit(X_nan, y_nan)
    r2 = dt3.score(X_nan, y_nan)
    mae = mean_absolute_error(y_nan, np.clip(dt3.predict(X_nan), 0, 1))
    label = str(d) if d else "None"
    print(f"  depth={label:5s}: R²={r2:.6f}  MAE={mae:.6f}")

# ── M-5: 비NaN 행에서 mean_working이 정수인가? 연속인가? ──
print("\n[M-5] mean_working 값 분포")
wk = train["mean_working"].dropna()
print(f"  unique 값 수: {wk.nunique()}")
print(f"  값 목록: {sorted(wk.unique())}")
print(f"  정수 여부: {(wk == wk.round()).all()}")

[M-1] NaN 행 vs 비NaN 행 — Full Tree 각각 학습
  비NaN 행 (n=1968): train MAE=0.000000
  비NaN 행 R²=1.000000
  비NaN 행 리프 수: 1560

  NaN 행 (n=1032): train MAE=0.000000
  NaN 행 R²=1.000000
  NaN 행 리프 수: 745

[M-2] 비NaN 행에서 mean_working 제거 시 R²
  mean_working 제거: R²=1.000000  MAE=0.000000
  (R²=1.000이면 mean_working이 없어도 완전 결정됨)
  (R²<1.000이면 mean_working이 공식의 핵심 인자)

[M-3] 비NaN 행 depth별 R² (mean_working 포함)
  depth=5    : R²=0.122268  MAE=0.232394
  depth=10   : R²=0.391200  MAE=0.170020
  depth=15   : R²=0.760732  MAE=0.079270
  depth=20   : R²=0.950662  MAE=0.018959
  depth=25   : R²=0.991868  MAE=0.003409
  depth=30   : R²=0.999981  MAE=0.000061
  depth=None : R²=1.000000  MAE=0.000000

[M-4] NaN 행 depth별 R² (mean_working 없이 수치 8개만)
  depth=5    : R²=0.131300  MAE=0.219350
  depth=10   : R²=0.440062  MAE=0.152885
  depth=15   : R²=0.744118  MAE=0.078165
  depth=20   : R²=0.961513  MAE=0.017019
  depth=25   : R²=0.999973  MAE=0.000132
  depth=30   : R²=1.000000  MAE=0.000000
  depth=None : R²=1.0

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
import warnings

warnings.filterwarnings("ignore")

train = pd.read_csv("../../train.csv")
test = pd.read_csv("../../test.csv")
sub = pd.read_csv("../../sample_submission.csv")
y = train["stress_score"]

# ── 핵심 피처: 수치 8개만 ──
FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

X_train = train[FEATS].values
X_test = test[FEATS].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 55)
print("실험 N: 수치 8개만으로 RandomForest / ExtraTrees OOF")
print("=" * 55)

# N-1: RandomForest (트리 수, depth 변화)
print("\n[RandomForest]")
for n_est, max_d in [(100, None), (300, None), (500, None), (300, 30), (300, 25)]:
    oof = np.zeros(len(X_train))
    for tr_i, va_i in kf.split(X_train):
        m = RandomForestRegressor(
            n_estimators=n_est, max_depth=max_d, n_jobs=-1, random_state=42
        )
        m.fit(X_train[tr_i], y.values[tr_i])
        oof[va_i] = np.clip(m.predict(X_train[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    print(f"  n_est={n_est:4d}  max_depth={str(max_d):5s}: OOF MAE={mae:.5f}")

# N-2: ExtraTrees (더 깊은 랜덤 분할)
print("\n[ExtraTrees]")
for n_est, max_d in [(300, None), (500, None)]:
    oof = np.zeros(len(X_train))
    for tr_i, va_i in kf.split(X_train):
        m = ExtraTreesRegressor(
            n_estimators=n_est, max_depth=max_d, n_jobs=-1, random_state=42
        )
        m.fit(X_train[tr_i], y.values[tr_i])
        oof[va_i] = np.clip(m.predict(X_train[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    print(f"  n_est={n_est:4d}  max_depth={str(max_d):5s}: OOF MAE={mae:.5f}")

# N-3: 가장 좋은 설정으로 전체 학습 + test 예측 + 제출
print("\n[최종 제출: RF n=500, depth=None, 수치 8개만]")
rf_final = RandomForestRegressor(
    n_estimators=500, max_depth=None, n_jobs=-1, random_state=42
)
rf_final.fit(X_train, y)

train_mae = mean_absolute_error(y, np.clip(rf_final.predict(X_train), 0, 1))
test_pred = np.round(np.clip(rf_final.predict(X_test), 0, 1) * 100) / 100

print(f"  train MAE: {train_mae:.5f}")
print(f"  test 분포: mean={test_pred.mean():.4f}  std={test_pred.std():.4f}")

import os

os.makedirs("result/v12", exist_ok=True)
sub["stress_score"] = test_pred
sub.to_csv("result/v12/submit_v12_RF.csv", index=False)
print("  저장: result/v12/submit_v12_RF.csv")

# N-4: 피처 중요도
print("\n[피처 중요도]")
for name, imp in sorted(zip(FEATS, rf_final.feature_importances_), key=lambda x: -x[1]):
    bar = "█" * int(imp * 100)
    print(f"  {name:35s}: {imp:.4f}  {bar}")

실험 N: 수치 8개만으로 RandomForest / ExtraTrees OOF

[RandomForest]
  n_est= 100  max_depth=None : OOF MAE=0.19538
  n_est= 300  max_depth=None : OOF MAE=0.19451
  n_est= 500  max_depth=None : OOF MAE=0.19452
  n_est= 300  max_depth=30   : OOF MAE=0.19483
  n_est= 300  max_depth=25   : OOF MAE=0.19588

[ExtraTrees]
  n_est= 300  max_depth=None : OOF MAE=0.16324
  n_est= 500  max_depth=None : OOF MAE=0.16297

[최종 제출: RF n=500, depth=None, 수치 8개만]
  train MAE: 0.06619
  test 분포: mean=0.4936  std=0.1255
  저장: result/v12/submit_v12_RF.csv

[피처 중요도]
  cholesterol                        : 0.1511  ███████████████
  height                             : 0.1435  ██████████████
  glucose                            : 0.1434  ██████████████
  weight                             : 0.1409  ██████████████
  bone_density                       : 0.1160  ███████████
  systolic_blood_pressure            : 0.1127  ███████████
  diastolic_blood_pressure           : 0.1062  ██████████
  age                          

In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
import warnings

warnings.filterwarnings("ignore")

# train = pd.read_csv('train.csv')
# test  = pd.read_csv('test.csv')
# sub   = pd.read_csv('sample_submission.csv')
y = train["stress_score"]

FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

X_train = train[FEATS].values
X_test = test[FEATS].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 55)
print("실험 O: ExtraTrees 정밀 튜닝")
print("목표: test std를 0.28에 가깝게, OOF MAE를 0.13 이하로")
print("=" * 55)

# O-1: min_samples_leaf 조정 (작을수록 더 세밀하게 분할)
print("\n[min_samples_leaf 변화 — 핵심 파라미터]")
for msl in [1, 2, 3, 5]:
    oof = np.zeros(len(X_train))
    for tr_i, va_i in kf.split(X_train):
        m = ExtraTreesRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=msl,
            n_jobs=-1,
            random_state=42,
        )
        m.fit(X_train[tr_i], y.values[tr_i])
        oof[va_i] = np.clip(m.predict(X_train[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    std = oof.std()
    print(f"  min_samples_leaf={msl}: OOF MAE={mae:.5f}  pred_std={std:.4f}")

# O-2: max_features 조정 (전체 피처 사용 vs 일부)
print("\n[max_features 변화]")
for mf in [1.0, 0.8, 0.6, "sqrt"]:
    oof = np.zeros(len(X_train))
    for tr_i, va_i in kf.split(X_train):
        m = ExtraTreesRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            max_features=mf,
            n_jobs=-1,
            random_state=42,
        )
        m.fit(X_train[tr_i], y.values[tr_i])
        oof[va_i] = np.clip(m.predict(X_train[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    std = oof.std()
    print(f"  max_features={str(mf):6s}: OOF MAE={mae:.5f}  pred_std={std:.4f}")

# O-3: 트리 수 대폭 증가 (500→2000)
print("\n[트리 수 증가]")
for n_est in [500, 1000, 2000]:
    oof = np.zeros(len(X_train))
    for tr_i, va_i in kf.split(X_train):
        m = ExtraTreesRegressor(
            n_estimators=n_est,
            max_depth=None,
            min_samples_leaf=1,
            max_features=1.0,
            n_jobs=-1,
            random_state=42,
        )
        m.fit(X_train[tr_i], y.values[tr_i])
        oof[va_i] = np.clip(m.predict(X_train[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    std = oof.std()
    print(f"  n_estimators={n_est:5d}: OOF MAE={mae:.5f}  pred_std={std:.4f}")

# O-4: ExtraTrees + mean_working 정수값 추가 시 변화
print("\n[mean_working 정수값 추가 효과]")
for use_wk in [False, True]:
    feats = FEATS + (["mean_working"] if use_wk else [])
    train_sub = train.copy()
    if use_wk:
        # NaN을 별도 값(-1)으로 처리
        train_sub["mean_working"] = train_sub["mean_working"].fillna(-1)
        test_sub = test.copy()
        test_sub["mean_working"] = test_sub["mean_working"].fillna(-1)
        Xtr = train_sub[feats].values
        Xte = test_sub[feats].values
    else:
        Xtr = train_sub[FEATS].values
        Xte = test[FEATS].values

    oof = np.zeros(len(Xtr))
    for tr_i, va_i in kf.split(Xtr):
        m = ExtraTreesRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            max_features=1.0,
            n_jobs=-1,
            random_state=42,
        )
        m.fit(Xtr[tr_i], y.values[tr_i])
        oof[va_i] = np.clip(m.predict(Xtr[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    print(f"  mean_working 포함={str(use_wk):5s}: OOF MAE={mae:.5f}")

# O-5: 최고 설정으로 제출 파일 생성
print("\n[최종 제출: ExtraTrees 최적 설정]")
best_model = ExtraTreesRegressor(
    n_estimators=2000,
    max_depth=None,
    min_samples_leaf=1,
    max_features=1.0,
    n_jobs=-1,
    random_state=42,
)
best_model.fit(X_train, y)

test_pred = np.round(np.clip(best_model.predict(X_test), 0, 1) * 100) / 100
train_pred = np.clip(best_model.predict(X_train), 0, 1)

print(f"  train MAE: {mean_absolute_error(y, train_pred):.5f}")
print(f"  test 분포: mean={test_pred.mean():.4f}  std={test_pred.std():.4f}")
print(f"  train 분포: mean={y.mean():.4f}  std={y.std():.4f}")

import os

os.makedirs("result/v12", exist_ok=True)
sub["stress_score"] = test_pred
sub.to_csv("result/v12/submit_v12_ET.csv", index=False)
print("  저장: result/v12/submit_v12_ET.csv")

실험 O: ExtraTrees 정밀 튜닝
목표: test std를 0.28에 가깝게, OOF MAE를 0.13 이하로

[min_samples_leaf 변화 — 핵심 파라미터]
  min_samples_leaf=1: OOF MAE=0.16297  pred_std=0.1719
  min_samples_leaf=2: OOF MAE=0.18726  pred_std=0.1298
  min_samples_leaf=3: OOF MAE=0.20556  pred_std=0.0991
  min_samples_leaf=5: OOF MAE=0.22399  pred_std=0.0685

[max_features 변화]
  max_features=1.0   : OOF MAE=0.16297  pred_std=0.1719
  max_features=0.8   : OOF MAE=0.16349  pred_std=0.1700
  max_features=0.6   : OOF MAE=0.16434  pred_std=0.1672
  max_features=sqrt  : OOF MAE=0.16556  pred_std=0.1627

[트리 수 증가]
  n_estimators=  500: OOF MAE=0.16297  pred_std=0.1719
  n_estimators= 1000: OOF MAE=0.16291  pred_std=0.1717
  n_estimators= 2000: OOF MAE=0.16281  pred_std=0.1716

[mean_working 정수값 추가 효과]
  mean_working 포함=False: OOF MAE=0.16297
  mean_working 포함=True : OOF MAE=0.18695

[최종 제출: ExtraTrees 최적 설정]
  train MAE: 0.00000
  test 분포: mean=0.4967  std=0.1817
  train 분포: mean=0.4821  std=0.2883
  저장: result/v12/submit_v12_ET.csv


In [6]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
import warnings

warnings.filterwarnings("ignore")

# train = pd.read_csv('train.csv')
# test  = pd.read_csv('test.csv')
# sub   = pd.read_csv('sample_submission.csv')
y = train["stress_score"]

FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

X_train = train[FEATS].values
X_test = test[FEATS].values

# ═══════════════════════════════════════════════════════
# 실험 P: 각 피처를 구간화(quantile bin)한 후
#         구간 조합으로 stress 평균을 직접 계산
# 핵심 아이디어: 공식이 각 피처의 구간 조합으로 결정된다면
#               충분히 세밀한 bin에서 그룹 평균 = 공식값에 수렴
# ═══════════════════════════════════════════════════════
print("=" * 55)
print("실험 P: Quantile Binning 그룹 평균")
print("=" * 55)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for n_bins in [5, 8, 10, 15, 20]:
    train_b = train[FEATS].copy()
    test_b = test[FEATS].copy()

    # 각 피처를 n_bins 분위수로 구간화 (train 기준)
    bin_cols = []
    for c in FEATS:
        col = f"{c}_bin"
        train_b[col] = pd.qcut(train_b[c], q=n_bins, labels=False, duplicates="drop")
        # test는 train의 분위수 경계를 그대로 사용
        quantiles = train_b[c].quantile(np.linspace(0, 1, n_bins + 1)).unique()
        test_b[col] = pd.cut(test[c], bins=quantiles, labels=False, include_lowest=True)
        bin_cols.append(col)

    # 그룹 키 생성
    train_b["combo"] = (
        train_b[bin_cols].astype(str).apply(lambda r: "_".join(r), axis=1)
    )
    test_b["combo"] = test_b[bin_cols].astype(str).apply(lambda r: "_".join(r), axis=1)

    # OOF: fold별로 train 그룹 평균 → val 예측
    oof = np.full(len(train), np.nan)
    train_b["stress"] = y.values

    for tr_i, va_i in kf.split(X_train):
        tr_df = train_b.iloc[tr_i]
        va_df = train_b.iloc[va_i]
        group_mean = tr_df.groupby("combo")["stress"].mean()
        global_mean = tr_df["stress"].mean()

        preds = va_df["combo"].map(group_mean).fillna(global_mean)
        oof[va_i] = np.clip(preds.values, 0, 1)

    oof_mae = mean_absolute_error(y, oof)
    coverage = (~np.isnan(oof)).mean()
    print(f"  n_bins={n_bins:2d}: OOF MAE={oof_mae:.5f}  " f"pred_std={oof.std():.4f}")

# ═══════════════════════════════════════════════════════
# 실험 Q: 각 피처를 개별 구간화 + 피처별 독립 점수 합산
# 공식이 weighted sum of individual feature scores라면
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("실험 Q: 피처별 단조 함수 추정")
print("각 피처를 20구간으로 나누고 구간별 stress 평균 계산")
print("→ 공식의 각 피처 기여도를 직접 시각화")
print("=" * 55)

for feat in FEATS:
    x = train[feat]
    bins = pd.qcut(x, q=20, duplicates="drop")
    stats = train.groupby(bins, observed=True)["stress_score"].agg(
        ["mean", "std", "count"]
    )

    # 단조성 확인 (오름/내림 여부)
    means = stats["mean"].values
    is_monotone_up = all(means[i] <= means[i + 1] for i in range(len(means) - 1))
    is_monotone_down = all(means[i] >= means[i + 1] for i in range(len(means) - 1))

    trend = (
        "↑단조증가"
        if is_monotone_up
        else ("↓단조감소" if is_monotone_down else "비단조")
    )
    span = means.max() - means.min()
    print(f"\n  [{feat}] {trend}  span={span:.4f}")
    for idx, row in stats.iterrows():
        bar = "█" * int(row["mean"] * 20)
        print(f"    {str(idx):30s}: mean={row['mean']:.3f}  {bar}")

# ═══════════════════════════════════════════════════════
# 실험 R: stress_score가 0.01 단위 정수인가?
#         → 공식이 이산값을 생성하는지 확인
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("실험 R: stress_score 이산값 구조 분석")
print("=" * 55)

y_vals = y.values
print(f"  unique 값 수: {len(np.unique(y_vals))}")
print(f"  0.01 단위 정수 여부: {np.all(np.round(y_vals*100) == y_vals*100)}")
print(f"  0.01 단위 unique 수: {len(np.unique(np.round(y_vals*100)/100))}")

# 각 unique 값의 빈도
val_counts = pd.Series(np.round(y_vals * 100) / 100).value_counts().sort_index()
print(f"\n  값 분포 (상위 20개):")
for v, c in val_counts.head(20).items():
    print(f"    {v:.2f}: {c}건")
print(f"  ...")
print(f"  (하위 20개)")
for v, c in val_counts.tail(20).items():
    print(f"    {v:.2f}: {c}건")

실험 P: Quantile Binning 그룹 평균
  n_bins= 5: OOF MAE=0.17084  pred_std=0.1723
  n_bins= 8: OOF MAE=0.17312  pred_std=0.1608
  n_bins=10: OOF MAE=0.18171  pred_std=0.1490
  n_bins=15: OOF MAE=0.19118  pred_std=0.1407
  n_bins=20: OOF MAE=0.20332  pred_std=0.1250

실험 Q: 피처별 단조 함수 추정
각 피처를 20구간으로 나누고 구간별 stress 평균 계산
→ 공식의 각 피처 기여도를 직접 시각화

  [age] 비단조  span=0.1043
    (16.999, 21.0]                : mean=0.488  █████████
    (21.0, 25.0]                  : mean=0.486  █████████
    (25.0, 28.0]                  : mean=0.463  █████████
    (28.0, 32.0]                  : mean=0.449  ████████
    (32.0, 35.0]                  : mean=0.494  █████████
    (35.0, 39.0]                  : mean=0.519  ██████████
    (39.0, 42.0]                  : mean=0.485  █████████
    (42.0, 45.6]                  : mean=0.449  ████████
    (45.6, 49.0]                  : mean=0.490  █████████
    (49.0, 53.0]                  : mean=0.415  ████████
    (53.0, 56.0]                  : mean=0.470  █████████
  

In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore")

# train = pd.read_csv('train.csv')
# test  = pd.read_csv('test.csv')
# sub   = pd.read_csv('sample_submission.csv')
y = train["stress_score"].values

FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

# ═══════════════════════════════════════════════════════
# 실험 S: Percentile Rank 선형 결합
# 핵심 아이디어: stress = weighted sum of percentile ranks
# ═══════════════════════════════════════════════════════
print("=" * 55)
print("실험 S: Percentile Rank 선형 결합")
print("=" * 55)

# S-1: train 전체 기준 percentile rank 계산
# (Data Leakage 주의: 실제 제출 시에는 train만으로 rank 계산)
all_data = pd.concat([train[FEATS], test[FEATS]], keys=["train", "test"])

ranks_train = pd.DataFrame()
ranks_test = pd.DataFrame()

for c in FEATS:
    # train 내 percentile rank (0~1)
    ranks_train[c] = train[c].rank(pct=True)
    # test: train 분포 기준으로 보간
    from scipy.stats import percentileofscore

    ranks_test[c] = test[c].apply(
        lambda v: percentileofscore(train[c].values, v, kind="rank") / 100
    )

X_rank_train = ranks_train.values
X_rank_test = ranks_test.values

# S-2: 단순 평균 (균등 가중치)
pred_equal = X_rank_train.mean(axis=1)
mae_equal = mean_absolute_error(y, pred_equal)
print(f"\n[균등 가중치] MAE={mae_equal:.5f}  " f"pred_std={pred_equal.std():.4f}")


# S-3: 가중치 최적화 (SLSQP)
def objective(w):
    w = np.abs(w) / np.abs(w).sum()
    pred = X_rank_train @ w
    return mean_absolute_error(y, np.clip(pred, 0, 1))


best_mae = 999
best_w = None
for _ in range(20):  # 여러 초기값으로 시도
    w0 = np.random.dirichlet(np.ones(len(FEATS)))
    res = minimize(objective, w0, method="SLSQP", options={"maxiter": 1000})
    if res.fun < best_mae:
        best_mae = res.fun
        best_w = np.abs(res.x) / np.abs(res.x).sum()

print(f"\n[최적화 가중치] MAE={best_mae:.5f}")
for name, w in zip(FEATS, best_w):
    print(f"  {name:35s}: {w:.4f}")

pred_opt = np.clip(X_rank_train @ best_w, 0, 1)
print(f"pred_std={pred_opt.std():.4f}")

# S-4: 각 피처의 방향성 탐색 (정방향 vs 역방향)
# 일부 피처는 높을수록 스트레스 감소일 수 있음
print("\n[방향성 탐색: 정방향(+) vs 역방향(-) 조합 최적화]")


def objective_signed(w):
    # w의 부호가 방향성, 절댓값이 가중치
    w_norm = w / np.abs(w).sum()
    # rank를 방향에 따라 변환
    X_dir = X_rank_train.copy()
    for i, wi in enumerate(w_norm):
        if wi < 0:
            X_dir[:, i] = 1 - X_rank_train[:, i]
    pred = X_dir @ np.abs(w_norm)
    return mean_absolute_error(y, np.clip(pred, 0, 1))


best_mae2 = 999
best_w2 = None
for _ in range(50):
    w0 = np.random.uniform(-1, 1, len(FEATS))
    res = minimize(
        objective_signed,
        w0,
        method="Nelder-Mead",
        options={"maxiter": 5000, "xatol": 1e-8},
    )
    if res.fun < best_mae2:
        best_mae2 = res.fun
        best_w2 = res.x

w2_norm = best_w2 / np.abs(best_w2).sum()
print(f"MAE={best_mae2:.5f}")
for name, w in zip(FEATS, w2_norm):
    direction = "(+정방향)" if w >= 0 else "(-역방향)"
    print(f"  {name:35s}: {abs(w):.4f} {direction}")

# S-5: OOF로 실제 일반화 성능 측정
print("\n[OOF 검증 — 최적화 가중치]")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(train))

for tr_i, va_i in kf.split(X_rank_train):
    # fold 내에서 rank 재계산 (Data Leakage 방지)
    X_fold_tr = pd.DataFrame()
    X_fold_va = pd.DataFrame()
    for c in FEATS:
        X_fold_tr[c] = train[c].iloc[tr_i].rank(pct=True).values
        X_fold_va[c] = (
            train[c]
            .iloc[va_i]
            .apply(
                lambda v: percentileofscore(train[c].iloc[tr_i].values, v, kind="rank")
                / 100
            )
            .values
        )

    # 가중치는 전체 train 기준으로 고정 (best_w 사용)
    oof[va_i] = np.clip(X_fold_va.values @ best_w, 0, 1)

oof_mae = mean_absolute_error(y, oof)
print(f"  OOF MAE={oof_mae:.5f}  pred_std={oof.std():.4f}")

# S-6: 제출 파일
print("\n[제출 파일 생성]")
pred_test = np.clip(X_rank_test.values @ best_w, 0, 1)
pred_test_r = np.round(pred_test * 100) / 100
print(f"  test 분포: mean={pred_test_r.mean():.4f}  std={pred_test_r.std():.4f}")

import os

os.makedirs("../result/v12", exist_ok=True)
sub["stress_score"] = pred_test_r
sub.to_csv("../result/v12/submit_v12_rank.csv", index=False)
print("  저장: ../result/v12/submit_v12_rank.csv")

실험 S: Percentile Rank 선형 결합

[균등 가중치] MAE=0.26262  pred_std=0.1180

[최적화 가중치] MAE=0.25146
  age                                : 0.4191
  height                             : 0.0075
  weight                             : 0.0160
  cholesterol                        : 0.0603
  systolic_blood_pressure            : 0.0081
  diastolic_blood_pressure           : 0.0385
  glucose                            : 0.0000
  bone_density                       : 0.4505
pred_std=0.0486

[방향성 탐색: 정방향(+) vs 역방향(-) 조합 최적화]
MAE=0.25038
  age                                : 0.3756 (-역방향)
  height                             : 0.0283 (-역방향)
  weight                             : 0.0471 (+정방향)
  cholesterol                        : 0.0698 (+정방향)
  systolic_blood_pressure            : 0.0367 (+정방향)
  diastolic_blood_pressure           : 0.0563 (+정방향)
  glucose                            : 0.0607 (-역방향)
  bone_density                       : 0.3255 (-역방향)

[OOF 검증 — 최적화 가중치]
  OOF MAE=0.25145  pred_std=0.0486


AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import warnings

warnings.filterwarnings("ignore")

# train = pd.read_csv('train.csv')
# test  = pd.read_csv('test.csv')
# sub   = pd.read_csv('sample_submission.csv')
y = train["stress_score"].values.astype(np.float32)

FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

X_train = train[FEATS].values.astype(np.float32)
X_test = test[FEATS].values.astype(np.float32)

# ═══════════════════════════════════════════════════════
# 실험 T: MLP (신경망) — 고차원 상호작용 포착
# ═══════════════════════════════════════════════════════
print("=" * 55)
print("실험 T: MLP 신경망")
print("=" * 55)


class StressNet(nn.Module):
    def __init__(self, n_feat, hidden, n_layers, dropout=0.0):
        super().__init__()
        layers = [nn.Linear(n_feat, hidden), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.ReLU()]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
        layers += [nn.Linear(hidden, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(
    X_tr,
    y_tr,
    X_va,
    y_va,
    hidden=256,
    n_layers=4,
    lr=1e-3,
    epochs=500,
    batch=256,
    dropout=0.0,
):
    scaler = StandardScaler()
    X_tr_s = torch.tensor(scaler.fit_transform(X_tr))
    X_va_s = torch.tensor(scaler.transform(X_va))
    y_tr_t = torch.tensor(y_tr)
    y_va_t = torch.tensor(y_va)

    model = StressNet(X_tr.shape[1], hidden, n_layers, dropout)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()

    best_val = 999
    best_state = None
    patience = 50
    no_improve = 0

    for ep in range(epochs):
        model.train()
        # mini-batch
        idx = torch.randperm(len(X_tr_s))
        for i in range(0, len(idx), batch):
            b = idx[i : i + batch]
            opt.zero_grad()
            loss = loss_fn(model(X_tr_s[b]), y_tr_t[b])
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_va_s).numpy()
            val_mae = mean_absolute_error(y_va, np.clip(val_pred, 0, 1))

        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_pred = np.clip(model(X_va_s).numpy(), 0, 1)
    return va_pred, best_val, scaler, model


# T-1: 구조별 OOF MAE
kf = KFold(n_splits=5, shuffle=True, random_state=42)

configs = [
    (128, 3, 1e-3, 0.0),
    (256, 4, 1e-3, 0.0),
    (512, 5, 1e-3, 0.0),
    (256, 6, 5e-4, 0.0),
    (512, 6, 5e-4, 0.1),
]

print("\n[MLP 구조별 OOF MAE]")
for hidden, n_layers, lr, drop in configs:
    oof = np.zeros(len(X_train))
    for tr_i, va_i in kf.split(X_train):
        va_pred, _, _, _ = train_mlp(
            X_train[tr_i],
            y[tr_i],
            X_train[va_i],
            y[va_i],
            hidden=hidden,
            n_layers=n_layers,
            lr=lr,
            epochs=1000,
            dropout=drop,
        )
        oof[va_i] = va_pred

    oof_mae = mean_absolute_error(y, oof)
    print(
        f"  hidden={hidden}  layers={n_layers}  lr={lr}  "
        f"drop={drop}: OOF MAE={oof_mae:.5f}  std={oof.std():.4f}"
    )

# T-2: 최고 설정으로 제출 파일 생성
# (위 결과 보고 후 결정하되, 우선 hidden=512, layers=6으로 시도)
print("\n[최종 학습 및 제출]")
scaler_final = StandardScaler()
X_tr_final = torch.tensor(scaler_final.fit_transform(X_train))
X_te_final = torch.tensor(scaler_final.transform(X_test))
y_tr_final = torch.tensor(y)

model_final = StressNet(8, 512, 6, dropout=0.0)
opt_final = optim.Adam(model_final.parameters(), lr=5e-4)
loss_fn = nn.L1Loss()

for ep in range(2000):
    model_final.train()
    idx = torch.randperm(len(X_tr_final))
    for i in range(0, len(idx), 256):
        b = idx[i : i + 256]
        opt_final.zero_grad()
        loss = loss_fn(model_final(X_tr_final[b]), y_tr_final[b])
        loss.backward()
        opt_final.step()

model_final.eval()
with torch.no_grad():
    tr_pred = np.clip(model_final(X_tr_final).numpy(), 0, 1)
    te_pred = np.clip(model_final(X_te_final).numpy(), 0, 1)

train_mae = mean_absolute_error(y, tr_pred)
te_pred_r = np.round(te_pred * 100) / 100

print(f"  train MAE: {train_mae:.5f}")
print(f"  test 분포: mean={te_pred_r.mean():.4f}  std={te_pred_r.std():.4f}")
print(f"  train 분포: mean={y.mean():.4f}  std={y.std():.4f}")

import os

os.makedirs("result/v12", exist_ok=True)
sub["stress_score"] = te_pred_r
sub.to_csv("result/v12/submit_v12_MLP.csv", index=False)
print("  저장: result/v12/submit_v12_MLP.csv")

실험 T: MLP 신경망

[MLP 구조별 OOF MAE]
  hidden=128  layers=3  lr=0.001  drop=0.0: OOF MAE=0.20146  std=0.2621
  hidden=256  layers=4  lr=0.001  drop=0.0: OOF MAE=0.18383  std=0.2459
  hidden=512  layers=5  lr=0.001  drop=0.0: OOF MAE=0.17789  std=0.2516
  hidden=256  layers=6  lr=0.0005  drop=0.0: OOF MAE=0.17893  std=0.2438
  hidden=512  layers=6  lr=0.0005  drop=0.1: OOF MAE=0.18173  std=0.2522

[최종 학습 및 제출]


KeyboardInterrupt: 

In [14]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from itertools import combinations
import warnings

warnings.filterwarnings("ignore")

# train = pd.read_csv("train.csv")
# test = pd.read_csv("test.csv")
y = train["stress_score"].values

FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ═══════════════════════════════════════════════════════
# 실험 U: 피처 부분집합 탐색
# 어떤 피처 조합이 가장 낮은 OOF MAE를 만드는가?
# depth=None Full Tree로 빠르게 탐색
# ═══════════════════════════════════════════════════════
print("=" * 55)
print("실험 U: 피처 부분집합 OOF MAE (Full Tree)")
print("=" * 55)

results = {}

# U-1: 피처 2개 조합 (8C2 = 28가지)
print("\n[2개 피처 조합 — 상위 10개]")
for feat_combo in combinations(FEATS, 2):
    X = train[list(feat_combo)].values
    oof = np.zeros(len(X))
    for tr_i, va_i in kf.split(X):
        dt = DecisionTreeRegressor(max_depth=None, random_state=42)
        dt.fit(X[tr_i], y[tr_i])
        oof[va_i] = np.clip(dt.predict(X[va_i]), 0, 1)
    results[feat_combo] = mean_absolute_error(y, oof)

top10_2 = sorted(results.items(), key=lambda x: x[1])[:10]
for combo, mae in top10_2:
    print(f"  {mae:.5f}  {combo}")

# U-2: 피처 3개 조합 (8C3 = 56가지)
print("\n[3개 피처 조합 — 상위 10개]")
results3 = {}
for feat_combo in combinations(FEATS, 3):
    X = train[list(feat_combo)].values
    oof = np.zeros(len(X))
    for tr_i, va_i in kf.split(X):
        dt = DecisionTreeRegressor(max_depth=None, random_state=42)
        dt.fit(X[tr_i], y[tr_i])
        oof[va_i] = np.clip(dt.predict(X[va_i]), 0, 1)
    results3[feat_combo] = mean_absolute_error(y, oof)

top10_3 = sorted(results3.items(), key=lambda x: x[1])[:10]
for combo, mae in top10_3:
    print(f"  {mae:.5f}  {combo}")

# U-3: 피처 4개 조합 (8C4 = 70가지)
print("\n[4개 피처 조합 — 상위 10개]")
results4 = {}
for feat_combo in combinations(FEATS, 4):
    X = train[list(feat_combo)].values
    oof = np.zeros(len(X))
    for tr_i, va_i in kf.split(X):
        dt = DecisionTreeRegressor(max_depth=None, random_state=42)
        dt.fit(X[tr_i], y[tr_i])
        oof[va_i] = np.clip(dt.predict(X[va_i]), 0, 1)
    results4[feat_combo] = mean_absolute_error(y, oof)

top10_4 = sorted(results4.items(), key=lambda x: x[1])[:10]
for combo, mae in top10_4:
    print(f"  {mae:.5f}  {combo}")

# U-4: 전체 8개 피처 OOF (기준값)
print("\n[전체 8개 피처 기준]")
X_all = train[FEATS].values
oof_all = np.zeros(len(X_all))
for tr_i, va_i in kf.split(X_all):
    dt = DecisionTreeRegressor(max_depth=None, random_state=42)
    dt.fit(X_all[tr_i], y[tr_i])
    oof_all[va_i] = np.clip(dt.predict(X_all[va_i]), 0, 1)
print(f"  OOF MAE={mean_absolute_error(y, oof_all):.5f}")

# U-5: 결정적 실험 — 피처를 하나씩 제거했을 때 OOF 변화
print("\n[피처 제거 실험 (Leave-One-Out)]")
for drop_feat in FEATS:
    feats_sub = [f for f in FEATS if f != drop_feat]
    X = train[feats_sub].values
    oof = np.zeros(len(X))
    for tr_i, va_i in kf.split(X):
        dt = DecisionTreeRegressor(max_depth=None, random_state=42)
        dt.fit(X[tr_i], y[tr_i])
        oof[va_i] = np.clip(dt.predict(X[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    delta = mae - mean_absolute_error(y, oof_all)
    direction = (
        "↑나빠짐" if delta > 0.001 else ("↓좋아짐" if delta < -0.001 else "변화없음")
    )
    print(f"  -{drop_feat:35s}: MAE={mae:.5f}  Δ={delta:+.5f}  {direction}")

실험 U: 피처 부분집합 OOF MAE (Full Tree)

[2개 피처 조합 — 상위 10개]
  0.22498  ('height', 'glucose')
  0.22524  ('glucose', 'bone_density')
  0.22732  ('height', 'bone_density')
  0.22790  ('weight', 'glucose')
  0.22800  ('cholesterol', 'bone_density')
  0.22837  ('weight', 'bone_density')
  0.22860  ('height', 'weight')
  0.22876  ('weight', 'cholesterol')
  0.23178  ('cholesterol', 'glucose')
  0.23232  ('height', 'cholesterol')

[3개 피처 조합 — 상위 10개]
  0.21135  ('height', 'weight', 'bone_density')
  0.21265  ('height', 'weight', 'glucose')
  0.21348  ('cholesterol', 'glucose', 'bone_density')
  0.21452  ('height', 'cholesterol', 'glucose')
  0.21469  ('height', 'glucose', 'bone_density')
  0.21510  ('height', 'weight', 'cholesterol')
  0.21519  ('weight', 'cholesterol', 'glucose')
  0.21621  ('weight', 'systolic_blood_pressure', 'glucose')
  0.21722  ('weight', 'cholesterol', 'bone_density')
  0.21754  ('weight', 'systolic_blood_pressure', 'bone_density')

[4개 피처 조합 — 상위 10개]
  0.20750  ('height'

In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
import warnings

warnings.filterwarnings("ignore")

# train = pd.read_csv("train.csv")
# test = pd.read_csv("test.csv")
# sub = pd.read_csv("sample_submission.csv")
y = train["stress_score"].values

FEATS = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

X_train = train[FEATS].values
X_test = test[FEATS].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ═══════════════════════════════════════════════════════
# 실험 V: KNN — 거리 기반 접근
# ExtraTrees가 0.163인데 KNN k=1이 더 나을 수 있음
# 핵심: 피처 스케일링 방법에 따라 거리 계산이 달라짐
# ═══════════════════════════════════════════════════════
print("=" * 55)
print("실험 V: KNN 정밀 탐색")
print("=" * 55)

scaler = StandardScaler()
X_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# V-1: k값별 OOF MAE
print("\n[k값별 OOF MAE — StandardScaler]")
for k in [1, 2, 3, 5, 7, 10, 15, 20]:
    oof = np.zeros(len(X_sc))
    for tr_i, va_i in kf.split(X_sc):
        knn = KNeighborsRegressor(
            n_neighbors=k, weights="distance" if k > 1 else "uniform"
        )
        knn.fit(X_sc[tr_i], y[tr_i])
        oof[va_i] = np.clip(knn.predict(X_sc[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    print(f"  k={k:2d}: OOF MAE={mae:.5f}  std={oof.std():.4f}")

# V-2: 피처 가중치 조정 — height, systolic_bp, glucose, bone_density 강조
print("\n[피처 가중치 조정 KNN k=1]")
# Leave-One-Out에서 중요도: height(0.010) > systolic_bp(0.011) > glucose(0.009) > bone_density(0.007)
weight_configs = [
    ("균등", [1, 1, 1, 1, 1, 1, 1, 1]),
    ("중요4개 강조", [0.5, 2, 0.5, 1, 2, 1, 2, 1.5]),
    ("height+sbp+gluc+bone", [0.3, 3, 0.3, 1, 3, 1, 3, 3]),
    ("상위4개만", [0, 1, 0, 0, 1, 0, 1, 1]),
]

from sklearn.preprocessing import StandardScaler

for name, feat_w in weight_configs:
    w = np.array(feat_w, dtype=float)
    X_w = X_train * w
    X_te_w = X_test * w
    sc2 = StandardScaler()
    X_w_sc = sc2.fit_transform(X_w)
    X_te_w_sc = sc2.transform(X_te_w)

    oof = np.zeros(len(X_w_sc))
    for tr_i, va_i in kf.split(X_w_sc):
        knn = KNeighborsRegressor(n_neighbors=1)
        knn.fit(X_w_sc[tr_i], y[tr_i])
        oof[va_i] = np.clip(knn.predict(X_w_sc[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    print(f"  [{name}]: OOF MAE={mae:.5f}  std={oof.std():.4f}")

# ═══════════════════════════════════════════════════════
# 실험 W: ExtraTrees + KNN 앙상블
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("실험 W: ExtraTrees + KNN 블렌딩")
print("=" * 55)

for alpha in [0.3, 0.5, 0.7, 0.9]:
    oof_et = np.zeros(len(X_train))
    oof_knn = np.zeros(len(X_train))

    for tr_i, va_i in kf.split(X_train):
        sc3 = StandardScaler()
        X_tr_s = sc3.fit_transform(X_train[tr_i])
        X_va_s = sc3.transform(X_train[va_i])

        et = ExtraTreesRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            n_jobs=-1,
            random_state=42,
        )
        et.fit(X_train[tr_i], y[tr_i])
        oof_et[va_i] = np.clip(et.predict(X_train[va_i]), 0, 1)

        knn = KNeighborsRegressor(n_neighbors=1)
        knn.fit(X_tr_s, y[tr_i])
        oof_knn[va_i] = np.clip(knn.predict(X_va_s), 0, 1)

    oof_blend = alpha * oof_et + (1 - alpha) * oof_knn
    mae = mean_absolute_error(y, np.clip(oof_blend, 0, 1))
    print(
        f"  ET×{alpha:.1f} + KNN×{1-alpha:.1f}: OOF MAE={mae:.5f}  std={oof_blend.std():.4f}"
    )

# ═══════════════════════════════════════════════════════
# 실험 X: 결정적 — train+test 피처 공간에서 KNN
# test 레이블은 사용 안 함, 피처만 활용해서 거리 계산
# (Semi-supervised: test 피처로 train 이웃 찾기)
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("실험 X: train+test 합친 피처 공간에서 KNN")
print("(test 레이블 미사용 — Data Leakage 없음)")
print("=" * 55)

X_all = np.vstack([X_train, X_test])
sc_all = StandardScaler()
X_all_sc = sc_all.fit_transform(X_all)
X_tr_sc_new = X_all_sc[: len(X_train)]
X_te_sc_new = X_all_sc[len(X_train) :]

for k in [1, 3, 5]:
    oof = np.zeros(len(X_tr_sc_new))
    for tr_i, va_i in kf.split(X_tr_sc_new):
        knn = KNeighborsRegressor(n_neighbors=k)
        knn.fit(X_tr_sc_new[tr_i], y[tr_i])
        oof[va_i] = np.clip(knn.predict(X_tr_sc_new[va_i]), 0, 1)
    mae = mean_absolute_error(y, oof)
    print(f"  train+test 스케일, k={k}: OOF MAE={mae:.5f}  std={oof.std():.4f}")

# ═══════════════════════════════════════════════════════
# 실험 Y: 각 test 포인트의 train 최근접 이웃 거리 분포
# 거리가 작을수록 → 해당 예측이 더 신뢰 가능
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("실험 Y: test 포인트의 train 최근접 거리 분포")
print("=" * 55)

from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=1)
nn.fit(X_tr_sc_new)
dists, _ = nn.kneighbors(X_te_sc_new)
dists = dists.flatten()

print(
    f"  최근접 거리: min={dists.min():.4f}  "
    f"mean={dists.mean():.4f}  "
    f"max={dists.max():.4f}  "
    f"std={dists.std():.4f}"
)
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  P{p:2d}: {np.percentile(dists, p):.4f}")

# train 내부 거리 (OOF 기준)
nn_tr = NearestNeighbors(n_neighbors=2)  # k=2 (자기 자신 제외)
nn_tr.fit(X_tr_sc_new)
dists_tr, _ = nn_tr.kneighbors(X_tr_sc_new)
dists_tr = dists_tr[:, 1]  # 두 번째 이웃 (자기 자신 제외)
print(
    f"\n  train 내부 최근접 거리: mean={dists_tr.mean():.4f}  "
    f"std={dists_tr.std():.4f}"
)
print(
    f"  test-train 거리 / train 내부 거리 비율: " f"{dists.mean()/dists_tr.mean():.3f}"
)
print("  (1.0에 가까울수록 test가 train과 동일한 밀도)")

실험 V: KNN 정밀 탐색

[k값별 OOF MAE — StandardScaler]
  k= 1: OOF MAE=0.19388  std=0.2872
  k= 2: OOF MAE=0.18658  std=0.2413
  k= 3: OOF MAE=0.18543  std=0.2159
  k= 5: OOF MAE=0.18301  std=0.1859
  k= 7: OOF MAE=0.18706  std=0.1685
  k=10: OOF MAE=0.19199  std=0.1506
  k=15: OOF MAE=0.19848  std=0.1311
  k=20: OOF MAE=0.20250  std=0.1188

[피처 가중치 조정 KNN k=1]
  [균등]: OOF MAE=0.19388  std=0.2872
  [중요4개 강조]: OOF MAE=0.19388  std=0.2872
  [height+sbp+gluc+bone]: OOF MAE=0.19388  std=0.2872
  [상위4개만]: OOF MAE=0.19030  std=0.2875

실험 W: ExtraTrees + KNN 블렌딩
  ET×0.3 + KNN×0.7: OOF MAE=0.17626  std=0.2405
  ET×0.5 + KNN×0.5: OOF MAE=0.16800  std=0.2132
  ET×0.7 + KNN×0.3: OOF MAE=0.16344  std=0.1911
  ET×0.9 + KNN×0.1: OOF MAE=0.16236  std=0.1761

실험 X: train+test 합친 피처 공간에서 KNN
(test 레이블 미사용 — Data Leakage 없음)
  train+test 스케일, k=1: OOF MAE=0.19373  std=0.2875
  train+test 스케일, k=3: OOF MAE=0.24870  std=0.1810
  train+test 스케일, k=5: OOF MAE=0.24625  std=0.1421

실험 Y: test 포인트의 train 최근접 거리 분포
 